In [2]:
# Install the Groq Python SDK
! pip install groq

In [3]:
import os
# Import the Groq client instead of OpenAI
from groq import Groq
# Used to securely store your API key
from google.colab import userdata

In [4]:
import os
from google.colab import userdata # Added import for userdata
# Get Groq API key from Colab secrets
groq_api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = groq_api_key

# Initialize the Groq client
client = Groq(api_key=groq_api_key)

def prompt_based_query(user_query):
    response = client.chat.completions.create(
        # Updated to a currently supported Groq model
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "you are a financial analyst"},
            {"role": "user", "content": user_query}
        ],
        temperature = 0.2
    )
    return response.choices[0].message.content

# Example of how to call the function (this line was outside the function previously)
# print(prompt_based_query('explain the revenue growth '))

In [5]:
import os
# Get Groq API key from Colab secrets
groq_api_key = userdata.get('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = groq_api_key

# Initialize the Groq client
client = Groq(api_key=groq_api_key)

def prompt_based_query(user_query):
    response = client.chat.completions.create(
        # Updated to a currently supported Groq model
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "you are a financial analyst"},
            {"role": "user", "content": user_query}
        ],
        temperature = 0.2
    )
    return response.choices[0].message.content

# Example of how to call the function (this line was outside the function previously)
# print(prompt_based_query('explain the revenue growth '))

In [6]:
# Install necessary libraries, including sentence-transformers for HuggingFaceEmbeddings
!pip install langchain faiss-cpu langchain_community tiktoken sentence-transformers

In [7]:
#step 1 load document
from langchain_community.document_loaders import TextLoader
loader=TextLoader('/content/Custom_Dataset_Finetune_Notebook.ipynb')
documents=loader.load()

/tmp/ipykernel_3598/2555071556.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [8]:
#step 2 create the embedding + vector DB
import os
# Import HuggingFaceEmbeddings as Groq does not provide an embedding model
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize HuggingFaceEmbeddings (a common choice for local embeddings)
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db=FAISS.from_documents(documents,embeddings)

/tmp/ipykernel_3598/3497621952.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
#step 3 Retrieval + Generation
def rag_query(query):
  docs=vector_db.similarity_search(query,k=3)
  context=" ".join(doc.page_content for doc in docs)
  response=client.chat.completions.create(
       # Updated to a currently supported Groq model
       model="llama-3.3-70b-versatile",
       messages=[
           {"role":"system","content":"use the provided context to answer"},
           {"role":"user","content": f"context:{context} \n\nQuestions:{query}"}
       ]
       )
  return response.choices[0].message.content

In [10]:
# Install libraries for fine-tuning
!pip install accelerate peft bitsandbytes transformers trl datasets

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from datasets import Dataset

In [12]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig # Added import for BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

# Configure 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
)

# Load a small base model suitable for Colab
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Load the model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto" # Ensure model is on GPU if available
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Set pad_token to eos_token for models that don't have a specific pad token (common for LLMs)
tokenizer.pad_token = tokenizer.eos_token

# Configure LoRA
lora_config = LoraConfig(
    r=8, # LoRA attention dimension
    lora_alpha=16, # Alpha parameter for LoRA scaling
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"], # Modules to apply LoRA to
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print the trainable parameters for verification
model.print_trainable_parameters()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.2044


In [13]:
# Prepare the dataset for causal language modeling

# Extract text content from the loaded documents
texts = [doc.page_content for doc in documents]

# Create a Hugging Face Dataset from the text
data_dict = {"text": texts}
dataset = Dataset.from_dict(data_dict)

# Define tokenization function
def tokenize_function(examples):
    # Truncate to maximum length of the model if necessary
    return tokenizer(examples["text"], truncation=True, max_length=tokenizer.model_max_length)

# Apply tokenization to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

In [14]:
print(rag_query("what is our company refund policy"))

The provided context does not mention a company refund policy. It appears to be a code snippet for loading a custom dataset from a JSONL file, and there is no information related to a refund policy. If you could provide more context or clarify which company you are referring to, I would be happy to try and assist you further.


In [15]:
#step 5: Training
from transformers import Trainer , TrainingArguments
#add labels to the tokenized_dataset for causal language mdelling
def add_labels_to_dataset(examples):
  examples['labels']=examples['input_ids']
  return examples
tokenized_dataset=tokenized_dataset.map(add_labels_to_dataset,batched=True)
training_args=TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    logging_steps=10,
    save_steps=50
)
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)
trainer.train()

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

Step,Training Loss


TrainOutput(global_step=2, training_loss=0.692449152469635, metrics={'train_runtime': 1.7627, 'train_samples_per_second': 1.135, 'train_steps_per_second': 1.135, 'total_flos': 2911236636672.0, 'train_loss': 0.692449152469635, 'epoch': 2.0})

In [16]:
#step 6: inferences
def generate_response(prompt):
  inputs= tokenizer(prompt, return_tensors="pt").to("cuda")
  outputs=model.generate(**inputs, max_new_tokens=100)
  return tokenizer.decode(outputs[0])

In [17]:
print(generate_response("What is ROI"))

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


<s> What is ROI and how can it be calculated?</s>
